In [2]:
# Basic imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime as dt

In [38]:
import joblib


## Getting Started
* Download the oct_2024.snappy.parquet and data_dict.csv files from the provided link
* data_dict.csv contains a description of every column in oct_2024.snappy.parquet
* oct_2024.snappy.parquet is a parquet file, which is a common file type, with numerous advantages over csv 
    * Compression
    * Schema preservation
    * Columnar format, you can read only the columns you need, very handy when you are dealing with very wide tables, or columns containing lots of data, such as text extracts or arrays and only want the associated metadata
    
## A Note on the Data
This dataset is constructed from multiple internal tables at Auto Trader with minimal cleaning, and so is representative of real data you would find out in the wild. This does mean that there are plenty of nulls and some oddities, for example there is one entry that has taken >4 years to sell. 

Good feature cleaning and engineering will be critical in achieving a good result. Some features have extremely high cardinality (e.g. derivative), you may want to find ways of reducing this, or finding alternative ways to represent the data.

Sometimes the data that is easy to pull is not the data that you want for the problem, think carefully about the features you have been provided and whether a production grade would would have access to them at inference time. Also consider any biases present in the dataset. This data is looking at a set of subset of sales that occured in October 2024, and is mainly last-seen properties of the advert.

In [7]:
# Data paths
dict_path = "data_dict.csv"
data_path = "oct_2024.snappy.parquet"

# Read in data
data_dict_df = pd.read_csv(dict_path)
raw_data_df = pd.read_parquet(data_path)

In [8]:
pd.set_option('display.max_colwidth', None) # Prevents truncation of long text columns
data_dict_df

,Field,Description
0,stock_item_id,Unique ID of the advert e.g. 8a42801e86dad35a0186dd181f956cb5
1,last_date_seen,The date the vehicle was last seen advertised on Auto Trader e.g. 2023-01-31
2,first_date_seen,The first date the vehicle was listed on Auto Trader
3,days_to_sell,The difference in days between first_date_seen and last_date_seen
4,first_retailer_asking_price,The price the advert had when first created. This may have been a temp holding price before it went live.
5,last_retailer_asking_price,The last seen price
6,can_home_deliver,This advert is able to be delivered to the buyers home
7,reviews_per_100_advertised_stock_last_12_months,"How many reviews the seller has recieved in the last 12 months per 100 stock advertised, i.e. 200 stock advertised over the whole year, 150 reviews 150/2= 75"
8,segment,Whether the seller is a Franchise or Independent retailer
9,seats,Number of seats in the vehicle


In [9]:
raw_data_df.head(4)

,stock_item_id,last_date_seen,first_date_seen,days_to_sell,first_retailer_asking_price,last_retailer_asking_price,can_home_deliver,reviews_per_100_advertised_stock_last_12_months,segment,seats,...,first_registration_date,attention_grabber,manufacturer_approved,price_indicator_rating,adjusted_retail_amount_gbp,predicted_mileage,number_of_images,first_image_label,advert_quality,postcode_area
0,33e3749b79a2847010ec1ad4716745393a2fd6b7d9db8d0563363085e1435502,2024-10-07,2022-10-12,726,24495,18995,False,0.3,Independent,5.0,...,2016-09-09,None,False,GOOD,18716.0,73842.0,11,FRONT_RIGHT,72,BT
1,cbf6f80e01e939aa017a9641d02f5cc24f78c3e69d9a742fe53edf30116053e6,2024-10-25,2023-05-13,531,16990,12990,False,0.9,Franchise,5.0,...,2021-01-14,£750 deposit contribution*,True,GOOD,13110.0,22740.0,12,FRONT_RIGHT,55,HA
2,c49040c0388b7533accbf7d4db144efbe7b2a251256b3bb3bdab3253f0e23abe,2024-10-05,2023-07-31,432,14481,11913,False,9.1,Independent,5.0,...,2016-12-17,SAT NAV • REAR CAMERA,False,GOOD,11873.0,55729.0,38,FRONT_RIGHT,41,BS
3,eacc77fecac6a43b328d0cbb16c192baa0654284a37b7d5219d81ff255124f5b,2024-10-21,2023-09-18,399,61000,36500,False,2.9,Franchise,5.0,...,2023-05-15,8.9 PERCENT APR AVAILABLE,False,GOOD,36809.0,10689.0,49,FRONT_RIGHT,71,PR


In [12]:
raw_data_df.shape

(155676, 43)

In [13]:
raw_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155676 entries, 0 to 155675
Data columns (total 43 columns):
 #   Column                                           Non-Null Count   Dtype  
---  ------                                           --------------   -----  
 0   stock_item_id                                    155676 non-null  object 
 1   last_date_seen                                   155676 non-null  object 
 2   first_date_seen                                  155676 non-null  object 
 3   days_to_sell                                     155676 non-null  int32  
 4   first_retailer_asking_price                      155676 non-null  int64  
 5   last_retailer_asking_price                       155676 non-null  int64  
 6   can_home_deliver                                 155676 non-null  bool   
 7   reviews_per_100_advertised_stock_last_12_months  141413 non-null  float64
 8   segment                                          155577 non-null  object 
 9   seats          

## Shuffle and Spilt Train/Test/Validate DataSet

In [35]:
def shuffle_and_split(df, random_seed=None):
    """
    Shuffle and split a DataFrame into train (80%), test (20%), 
    and further split train into training (90%) and validation (10%).

    Parameters:
    df (pd.DataFrame): The dataset to split
    random_seed (int, optional): Random seed for reproducibility

    Returns:
    tuple: (train_df, val_df, test_df)
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    # Shuffle the data
    shuffled_df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    # Define split sizes
    train_size = int(0.8 * len(df))
    val_size = int(0.1 * train_size)  # 10% of training set
    
    # Split data
    train_df = shuffled_df.iloc[:train_size]
    test_df = shuffled_df.iloc[train_size:]
    
    # Further split training into train and validation
    val_df = train_df.iloc[-val_size:]  # Last 10% for validation
    train_df = train_df.iloc[:-val_size]  # Remaining 90% for training
    
    return train_df, val_df, test_df


# Example Usage
raw_train_df, raw_val_df, raw_test_df = shuffle_and_split(raw_data_df, random_seed=42)

# Print sizes
print(f"Train size: {len(raw_train_df)}, Validation size: {len(raw_val_df)}, Test size: {len(raw_test_df)}")


Train size: 112086, Validation size: 12454, Test size: 31136


### Save train/test/val for reuse

In [44]:
# raw_train_df.to_parquet("datasets/raw_train_oct_2024.snappy.parquet", compression="snappy", index=False)
# raw_val_df.to_parquet("datasets/raw_val_oct_2024.snappy.parquet", compression="snappy", index=False)
# raw_test_df.to_parquet("datasets/raw_test_oct_2024.snappy.parquet", compression="snappy", index=False)

print("All datasets saved successfully!")

All datasets saved successfully!
